# 06.8 · Anomaly detection: z-score, IQR, Isolation Forest

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/08-anomaly-detection/lesson.ipynb)

Read the [README](README.md) first and do the 10-order calculation on paper. This notebook repeats it, shows isolation by random cuts, and then hunts for odd invoices in a year of ShopWave orders.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## 1. Ten orders, by hand

In [2]:
orders = np.array([42, 55, 38, 61, 47, 52, 44, 58, 49, 410], dtype=float)   # order values, £

mean, std = orders.mean(), orders.std()          # population std (ddof=0), as in the README
z = (orders - mean) / std
print(f"mean £{mean:.1f}, std £{std:.1f}")
print("z-scores:", z.round(2))
print("flagged by |z| > 3:", orders[np.abs(z) > 3])

mean £85.6, std £108.3
z-scores: [-0.4  -0.28 -0.44 -0.23 -0.36 -0.31 -0.38 -0.25 -0.34  2.99]
flagged by |z| > 3: []


The £410 order has z = 2.99 and is **not** flagged. It is so extreme that it drags the mean up (to £85.60) and inflates the standard deviation (to £108), hiding itself. This is called **masking**.

In [3]:
q1, q3 = np.percentile(orders, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 {q1}, Q3 {q3}, IQR {iqr}, fences ({low}, {high})")
print("flagged by IQR:", orders[(orders < low) | (orders > high)])

Q1 44.75, Q3 57.25, IQR 12.5, fences (26.0, 76.0)
flagged by IQR: [410.]


Quartiles barely move when one value goes wild, so the IQR rule catches the £410 order. (NumPy interpolates quartiles slightly differently from the paper method; the fence is 76 either way to the nearest pound.)

## 2. Isolation by random cuts

Isolation Forest's idea: pick a random cut point between the smallest and largest value, keep the side your point is on, repeat until it is alone. Odd points get isolated in very few cuts.

In [4]:
rng = np.random.default_rng(0)

def isolation_depth(values, target):
    """Cut at random points until `target` is alone. Return the number of cuts."""
    current, cuts = values.copy(), 0
    while len(current) > 1:
        cut = rng.uniform(current.min(), current.max())
        current = current[current < cut] if target < cut else current[current >= cut]
        cuts += 1
    return cuts

for v in orders:
    avg = np.mean([isolation_depth(orders, v) for _ in range(1000)])
    print(f"£{v:>5.0f}: isolated after {avg:4.2f} cuts on average")

£   42: isolated after 4.68 cuts on average
£   55: isolated after 4.93 cuts on average
£   38: isolated after 3.48 cuts on average
£   61: isolated after 3.65 cuts on average
£   47: isolated after 5.24 cuts on average
£   52: isolated after 5.05 cuts on average
£   44: isolated after 5.09 cuts on average
£   58: isolated after 4.48 cuts on average
£   49: isolated after 5.25 cuts on average
£  410: isolated after 1.05 cuts on average


The £410 order is isolated after about **1 cut**: almost any random cut between 38 and 410 lands to its left. Ordinary orders need 3.5 to 5. Notice that 38 and 61, the edges of the normal group, are also a bit easier to isolate than the middle values. An Isolation Forest does exactly this with many trees and many features, and turns average depth into a score.

In [5]:
from sklearn.ensemble import IsolationForest
m = IsolationForest(n_estimators=200, random_state=42).fit(orders.reshape(-1, 1))
scores = -m.score_samples(orders.reshape(-1, 1))      # higher = more anomalous
for v, s in zip(orders, scores):
    print(f"£{v:>5.0f}: anomaly score {s:.3f}")

£   42: anomaly score 0.428
£   55: anomaly score 0.414
£   38: anomaly score 0.529
£   61: anomaly score 0.504
£   47: anomaly score 0.400
£   52: anomaly score 0.406
£   44: anomaly score 0.409
£   58: anomaly score 0.436
£   49: anomaly score 0.394
£  410: anomaly score 0.821


Scores near 0.5 mean "ordinary"; scores well above 0.5 mean "isolated quickly". The £410 order stands alone at 0.82.

## 3. A year of ShopWave invoices

One row per invoice: total amount, number of items, average price per item, and hour of day.

In [6]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

392,693 clean transaction lines, 4,338 customers


In [7]:
inv = sales.groupby("Invoice").agg(
    customer=("Customer ID", "first"), country=("Country", "first"),
    when=("InvoiceDate", "first"), amount=("Revenue", "sum"),
    items=("Quantity", "sum"), lines=("StockCode", "nunique"), max_price=("Price", "max"))
inv["hour"] = inv["when"].dt.hour
inv["avg_price"] = inv["amount"] / inv["items"]
print(f"{len(inv):,} invoices")

inv[["amount", "items", "avg_price", "hour"]].describe().round(1)

18,532 invoices


,amount,items,avg_price,hour
count,18532.0,18532.0,18532.0,18532.0
mean,479.6,278.0,5.4,12.5
std,1678.1,972.4,91.9,2.4
min,0.4,1.0,0.1,6.0
25%,157.3,74.0,1.4,11.0
50%,302.6,154.0,1.9,12.0
75%,469.6,290.0,2.6,14.0
max,168469.6,80995.0,8142.8,20.0


In [8]:
from sklearn.ensemble import IsolationForest
features = ["amount", "items", "avg_price", "hour"]
X_inv = np.log1p(inv[features])                  # log: amounts and items are very skewed
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42).fit(X_inv)
inv["score"] = -iso.score_samples(X_inv)          # higher = easier to isolate = odder
inv["flag"] = iso.predict(X_inv) == -1
first_line = sales.groupby("Invoice").agg(code=("StockCode", "first"), description=("Description", "first"))
inv = inv.join(first_line)
print(f"flagged {inv['flag'].sum()} of {len(inv):,} invoices")

flagged 186 of 18,532 invoices


## 4. The ten oddest invoices

In [9]:
cols = ["customer", "country", "amount", "items", "avg_price", "hour", "code", "description", "score"]
inv.sort_values("score", ascending=False).head(10)[cols].round(2)

,customer,country,amount,items,avg_price,hour,code,description,score
Invoice,,,,,,,,,
551697,16029,United Kingdom,8142.75,1,8142.75,13,POST,POSTAGE,0.76
573080,12536,France,4161.06,1,4161.06,14,M,Manual,0.75
573077,12536,France,4161.06,1,4161.06,14,M,Manual,0.75
571751,12744,Singapore,6068.06,2,3034.03,11,M,Manual,0.75
569382,15502,United Kingdom,3155.95,1,3155.95,16,M,Manual,0.75
553018,12757,Portugal,557.72,1,557.72,19,M,Manual,0.74
562946,15581,United Kingdom,2500.00,1,2500.00,9,M,Manual,0.74
577130,15861,United Kingdom,165.00,1,165.00,19,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,0.74
536835,13145,United Kingdom,295.00,1,295.00,18,22655,VINTAGE RED KITCHEN CABINET,0.73


Read the `description` column. Almost none of these are suspicious customers:

- **Manual** (code `M`) lines are staff adjustments typed in by hand, often large round-ish amounts with a single "item".
- **POSTAGE** is a shipping charge, not a product.
- A couple are single, genuinely expensive pieces of furniture (a sideboard, a kitchen cabinet) in a shop that mostly sells £2 gifts.

What they share is a **shape**: one item, a high price per item. Isolation Forest finds unusual shapes. Whether an unusual shape is fraud, an error or just a big sale is a question for a human.

In [10]:
flagged = inv[inv["flag"]]
non_product = flagged["code"].isin(["M", "POST", "DOT", "C2", "BANK CHARGES", "AMAZONFEE", "D", "CRUK", "PADS"])
print(f"{non_product.mean():.0%} of flagged invoices are not product sales (manual, postage, fees)")
print(flagged["code"].value_counts().head(6))

44% of flagged invoices are not product sales (manual, postage, fees)
code
M               49
POST            26
22826            7
22655            5
22827            5
BANK CHARGES     3
Name: count, dtype: int64[pyarrow]


That is a **data-quality finding**: manual adjustments and postage are mixed in with product sales. They should be removed before calculating "revenue per customer" or training a model on products. Anomaly detection is often most useful exactly like this, as an automatic auditor.

What about the **biggest** invoice of the year?

In [11]:
biggest = inv["amount"].idxmax()
rank = (inv["score"] > inv.loc[biggest, "score"]).sum() + 1
print(inv.loc[biggest, ["amount", "items", "avg_price", "hour", "description", "score"]])
print(f"\nrank among {len(inv):,} invoices by anomaly score: {rank}")

amount                            168469.6
items                                80995
avg_price                             2.08
hour                                     9
description    PAPER CRAFT , LITTLE BIRDIE
score                             0.679818
Name: 581483, dtype: object

rank among 18,532 invoices by anomaly score: 95


£168,470 for 80,995 paper craft kits: extreme in total, but a normal price per item. Isolation Forest ranks it only around 95th. "Biggest" and "oddest" are different questions. If big orders are what you care about, a simple rule is clearer than a model.

## 5. The contamination setting

`contamination` does not change the scores, only where the cut-off is drawn. It is your guess of the share of anomalies, i.e. how many cases you can afford to review.

In [12]:
for c in [0.005, 0.01, 0.05]:
    n = (IsolationForest(n_estimators=200, contamination=c, random_state=42).fit(X_inv).predict(X_inv) == -1).sum()
    print(f"contamination {c:<5}: {n:>4} invoices flagged")

contamination 0.005:   93 invoices flagged


contamination 0.01 :  186 invoices flagged


contamination 0.05 :  927 invoices flagged


## 6. A second opinion: Local Outlier Factor

LOF asks a different question: is this point in a much **sparser** neighbourhood than its neighbours are? It looks for local oddities rather than global ones.

In [13]:
from sklearn.neighbors import LocalOutlierFactor
lof_flag = LocalOutlierFactor(n_neighbors=20, contamination=0.01).fit_predict(X_inv) == -1
both = (lof_flag & inv["flag"]).sum()
print(f"LOF flagged {lof_flag.sum()}, Isolation Forest flagged {inv['flag'].sum()}, both: {both}")

LOF flagged 186, Isolation Forest flagged 186, both: 28


Two reasonable methods agree on only a small part of their lists. Without labels there is no single right answer: each method has its own definition of "odd".

## 7. Evaluating without labels

You cannot compute precision or recall without knowing which invoices are truly bad. What you can do:

1. **Read the top of the list**, as above. Do the flags make sense to someone who knows the business?
2. **Check stability**: do different random seeds flag mostly the same invoices?
3. **Get a few labels**: send the top 50 to the finance team, record their verdicts, and measure precision on those.

In [14]:
sets = []
for seed in range(3):
    m = IsolationForest(n_estimators=200, contamination=0.01, random_state=seed).fit(X_inv)
    sets.append(set(inv.index[m.predict(X_inv) == -1]))
for s in [1, 2]:
    print(f"seed 0 vs seed {s}: {len(sets[0] & sets[s]) / len(sets[0]):.0%} of flags shared")

seed 0 vs seed 1: 82% of flags shared
seed 0 vs seed 2: 86% of flags shared


Around four in five flags survive a change of seed: the core of the list is stable, the edge is noisy.

## Quiz answers

1. z = (x − mean) / std: how many standard deviations a value sits from the mean. |z| > 3 is the usual flag.
2. One extreme value raises the mean and inflates the std so much that its own z-score falls below the threshold (masking). Quartiles are barely affected, so the IQR rule still catches it.
3. Random cuts isolate odd points quickly because they sit in empty space; average path length over many random trees becomes the anomaly score: short path, high score.
4. It sets the share of points labelled as anomalies, i.e. where the threshold on the score is drawn. It does not change the ranking.
5. Read and sanity-check the top of the list; check stability across seeds or methods; get a small labelled sample from domain experts and measure precision on it.

**Next:** [06.9 Market basket analysis](../09-market-basket/)